In [10]:
import pandas as pd

In [11]:
df = pd.read_csv("complaints_with_income.csv")

In [12]:
df.head()

,unique_key,created_date,closed_date,agency_name,complaint,complaint_detail,complaint_add_detail,zip,city,status,resol_descrip,resol_upd_date,borough,latitude,longitude,median_hh_income,pop,income_category,complaint_group
0,69197623,06/01/2026 02:07:09 PM,06/01/2026 02:16:05 PM,New York City Police Department,Illegal Parking,Blocked Sidewalk,NaN,11103,ASTORIA,Closed,The New York City Police Department responded ...,06/01/2026 02:16:08 PM,QUEENS,40.769581,-73.915593,93324.0,35163,Lower-middle income,parking and vehicles
1,69207017,06/01/2026 02:07:00 PM,06/01/2026 02:07:00 PM,Department of Environmental Protection,Water System,Hydrant Running (WC3),WC3,11209,BROOKLYN,Closed,The Department of Environmental Protection det...,06/01/2026 02:07:00 PM,BROOKLYN,40.624393,-74.038340,93854.0,71004,Lower-middle income,"drainage, water and plumbing"
2,69205534,06/01/2026 02:06:58 PM,06/01/2026 02:06:58 PM,Department of Buildings,Building/Use,No Certificate Of Occupancy/Illegal/Contrary T...,NaN,10456,BRONX,Open,Your Service Request has been submitted to the...,06/01/2026 12:00:00 AM,BRONX,40.833781,-73.913806,34954.0,87533,Lowest income,building/use
3,69198329,06/01/2026 02:06:52 PM,06/01/2026 02:43:43 PM,New York City Police Department,Illegal Parking,Blocked Bike Lane,NaN,11205,BROOKLYN,Closed,The New York City Police Department responded ...,06/01/2026 02:43:47 PM,BROOKLYN,40.697416,-73.956717,93887.0,51676,Lower-middle income,parking and vehicles
4,69196443,06/01/2026 02:06:51 PM,06/01/2026 04:50:30 PM,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,10026,NEW YORK,Closed,The New York City Police Department responded ...,06/01/2026 04:50:33 PM,MANHATTAN,40.802451,-73.946622,81244.0,37123,Lower-middle income,noise


In [13]:
df.shape

(3821055, 19)

In [14]:
#Checking income categories 
df['income_category'].value_counts(normalize=True)*100

income_category
Lowest income          46.349634
Lower-middle income    25.633470
Upper-middle income    14.790994
Highest income         13.225902
Name: proportion, dtype: float64

# Do richer neighbourhoods complaint more?

In [15]:
# Creating zip level dataset_one

df_zip = df.groupby("zip").agg(
    complaint_count=("unique_key", "count"),_
    pop=("pop", "first"),
    median_hh_income=("median_hh_income", "first"),
    income_category=("income_category", "first")
).reset_index()

SyntaxError: positional argument follows keyword argument (604563390.py, line 5)

In [ ]:
df_zip

In [ ]:
# creating complaints per 100K column 

df_zip["complaints_per_100k"] = (
    df_zip["complaint_count"] / df_zip["pop"] * 100000
)

In [ ]:
df_zip

In [ ]:
df_zip.groupby("income_category")["zip"].count() 
# So the comparison is fair at the ZIP level, we have equal zips across income levels

In [ ]:
df_zip.groupby("income_category")["complaints_per_100k"].mean()

In [ ]:
df_zip.groupby("income_category")["complaints_per_100k"].median()

**Richer ZIP codes do not appear to complain more after adjusting for population. In this analysis, lower-income ZIPs have higher 311 complaint rates per 100,000 residents. This does not prove that lower-income residents complain more; it could also reflect more 311-reportable problems in those ZIPs.**

# How do complaint types differ between lower-income and higher-income ZIPs?

In [ ]:
complaint_by_income = (
    df
    .groupby("income_category")["complaint_group"]
    .value_counts(normalize=True)
    .mul(100)
    .reset_index(name="percent")
)

complaint_by_income.head(20)

In [ ]:
top10 = (
    complaint_by_income
    .groupby("income_category")
    .head(10)
)

top10

**noise, drainage water and plumbing, and parking and vehicles, dirty/unsanitary conditions are the top four categories across all income groups.**

* **Noise complaints** make up **24.2%** of complaints in the **lowest-income ZIPs**, compared with **18.2%** in the **highest-income ZIPs** and **14.5–16.7%** in the middle-income groups.

* **Drainage, water and plumbing** complaints account for **20.0%** of complaints in the **lowest-income ZIPs**, more than double the share in the **highest-income ZIPs** (**9.0%**) and substantially higher than in the middle-income groups (**10.7–15.0%**).

* **Parking and vehicle** complaints show the opposite pattern. They are the largest complaint category in the middle-income groups (**30.1%** in lower-middle and **27.5%** in upper-middle ZIPs), compared with **19.8%** in the lowest-income and **17.6%** in the highest-income ZIPs.

* **Dirty or unsanitary conditions** are relatively stable across income groups, accounting for **5.3–7.5%** of complaints regardless of income.

* Several complaint types are much more prominent in **higher-income ZIPs**, including **encampment (5.3%)**, **homeless person assistance (3.2%)**, **vendor enforcement (3.1%)**, and **noise – helicopter (2.5%)**. None of these appear among the top complaints in the lower-income groups.

# How do response times by the city differ across rich and poorer neighbourhoods?

In [ ]:
df["created_date"] = pd.to_datetime(
    df["created_date"],
    format="%m/%d/%Y %I:%M:%S %p"
)

df["closed_date"] = pd.to_datetime(
    df["closed_date"],
    format="%m/%d/%Y %I:%M:%S %p"
)

In [ ]:
df['created_date'].dtype
df['closed_date'].dtype
# now they are in the right format

In [ ]:
#creating another column called response_days
df["response_days"] = ((df["closed_date"] - df["created_date"]).dt.total_seconds())/ 86400

### Noise

In [ ]:
# checking that all noise is handled by same agency or not
df_noise = df[df['complaint_group'] == 'noise']

In [ ]:
df_noise["agency_name"].value_counts(normalize=True).cumsum()
# so it should be reasonable to check for response time to noise complaint within the broader category. 

In [ ]:
df[df["complaint_group"] == "noise"] \
    .groupby("agency_name")["response_days"] \
    .median()

In [ ]:
(
    df[df["complaint_group"] == "noise"]
    .groupby("income_category")["agency_name"]
    .value_counts(normalize=True)
    .mul(100)
)

In [ ]:
#separating noise from environmental protection dept 

df_noise_nypd = df[
    (df["complaint_group"] == "noise") &
    (df["agency_name"] == "New York City Police Department")
].copy()

In [ ]:
df_noise_nypd.groupby('income_category')['response_days'].median()*24

**Among NYPD-handled noise complaints, median response times are slightly faster in the highest-income ZIP codes (0.68 hours) than in the other income groups (0.97–1.15 hours). However, the differences are relatively small—measured in minutes rather than days—and could reflect factors such as patrol locations, call volume, or geographic differences rather than differences in service quality. This analysis alone does not provide evidence of systematic bias in response times.**

### parking and vehicles

In [ ]:
# checking that all noise is handled by same agency or not
df_parking = df[df['complaint_group'] == 'parking and vehicles']

In [ ]:
df_parking["agency_name"].value_counts(normalize=True).cumsum()
# so it should be reasonable to check for response time to noise complaint within the broader category. 

In [ ]:
df[df["complaint_group"] == "parking and vehicles"] \
    .groupby("agency_name")["response_days"] \
    .median()*24

In [ ]:
(
    df[df["complaint_group"] == "parking and vehicles"]
    .groupby("income_category")["agency_name"]
    .value_counts(normalize=True)
    .mul(100)
)

In [ ]:
#separating noise from environmental protection dept 

df_parking_nypd = df[
    (df["complaint_group"] == "parking and vehicles") &
    (df["agency_name"] == "New York City Police Department")
].copy()

df_parking_dos = df[
    (df["complaint_group"] == "parking and vehicles") &
    (df["agency_name"] == "Department of Sanitation")
].copy()

In [ ]:
df_parking_nypd.groupby('income_category')['response_days'].median()*24

In [ ]:
df_parking_dos.groupby('income_category')['response_days'].median()*24

**Again, the highest-income ZIPs receive slightly faster responses, but we're talking about a difference of roughly 40 minutes to 1 hour.**

### drainage, water, and plumbing

In [ ]:
# checking that all noise is handled by same agency or not
df_water = df[df['complaint_group'] == 'drainage, water and plumbing']

In [ ]:
df_water["agency_name"].value_counts(normalize=True).cumsum()
# so it should be reasonable to check for response time to noise complaint within the broader category. 

In [ ]:
#separating agencies

df_water_dhpd = df[
    (df["complaint_group"] == "drainage, water and plumbing") &
    (df["agency_name"] == "Department of Housing Preservation and Development")
].copy()

df_water_dep = df[
    (df["complaint_group"] == "drainage, water and plumbing") &
    (df["agency_name"] == "Department of Environmental Protection")
].copy()

In [ ]:
df_water_dhpd.groupby('income_category')['response_days'].median()*24

In [ ]:
df_water_dep.groupby('income_category')['response_days'].median()*24

***No clear pattern***

### dirty or unsanitary conditions

In [ ]:
# checking that all noise is handled by same agency or not
df_dirty = df[df['complaint_group'] == 'dirty or unsanitary conditions']

In [ ]:
df_dirty["agency_name"].value_counts(normalize=True)
# so it should be reasonable to check for response time to noise complaint within the broader category. 

In [ ]:
#separating agencies

df_dirty_ds = df[
    (df["complaint_group"] == "dirty or unsanitary conditions") &
    (df["agency_name"] == "Department of Sanitation")
].copy()

df_dirty_dhpd = df[
    (df["complaint_group"] == "dirty or unsanitary conditions") &
    (df["agency_name"] == "Department of Housing Preservation and Development")
].copy()

In [ ]:
df_dirty_ds.groupby('income_category')['response_days'].median()*24

In [ ]:
df_dirty_dhpd.groupby('income_category')['response_days'].median()*24

***Response times are strongly influenced by the type of complaint and the agency responsible for resolving it. After accounting for agency, some complaint types are resolved slightly faster in higher-income ZIPs, while others show little difference or even the opposite pattern.***

# NYPD

In [ ]:
df_nypd = df[
    df["agency_name"] == "New York City Police Department"
].copy()

df_nypd["complaint_group"].value_counts(normalize=True).mul(100).head(15)

### encampment

In [ ]:
# checking that all encampment complaints are handled by same agency or not
df_encamp = df[df['complaint_group'] == 'encampment']

In [ ]:
df_encamp["agency_name"].value_counts(normalize=True)

In [ ]:
#separating agencies

df_encamp_nypd = df[
    (df["complaint_group"] == "encampment") &
    (df["agency_name"] == "New York City Police Department")
].copy()

df_encamp_dhs = df[
    (df["complaint_group"] == "encampment") &
    (df["agency_name"] == "Department of Homeless Services")
].copy()

In [ ]:
df_encamp_nypd.groupby('income_category')['response_days'].median()*24

In [ ]:
df_encamp_dhs.groupby('income_category')['response_days'].median()*24

In [ ]:
df_encamp_dhs.groupby("income_category").size()

- ***Highest income zips get fastest responses for encampment***
- ***Encampment complaints are much larger in highest income categories -> complaints do differ for some reason in higher income neighbourhoods***

In [ ]:
agency_counts = (
    df
    .groupby(["complaint_group", "agency_name"])
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)

agency_counts.head(20)

In [ ]:
response_summary = (
    df
    .groupby(["complaint_group", "agency_name", "income_category"])
    .agg(
        n=("unique_key", "count"),
        median_hours=("response_days", lambda x: x.median() * 24),
        mean_hours=("response_days", lambda x: x.mean() * 24)
    )
    .reset_index()
)

response_summary.head(20)